# ISLP Chapter 5 Lab
#### Author: Thomas Fitzgerald
This notebook contains my work for the lab accompanying Chapter 5 of [*An Introduction to Statistical Learning with Python* ("*ISLP*")](https://www.statlearning.com/). The fifth chapter of *ISLP* focuses on the use of resamlpling methods for analysis of models (specifically cross-validation and the bootstrap). This lab makes use of libraries introduced in the previous lab (**numpy**, **statsmodel**, **ISLP**, and **sklearn**) as well as portions of the **functools** library. To begin, the imports from the previous labs are pulled in:

In [1]:
import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (
    ModelSpec as MS,
    summarize,
    poly
)
from sklearn.model_selection import train_test_split

Next, new imports are pulled:

In [2]:
from functools import partial
from sklearn.model_selection import (
    cross_validate,
    KFold,
    ShuffleSplit
)
from sklearn.base import clone
from ISLP.models import sklearn_sm

### The Validation Set Approach
In this section, the validation set approach is used to estimate the test error rate of a set of models on the **Auto** data set. The **train_test_split()** function is used to split the data into validation sets. To begin, two sets are used. These sets are created with a random seed to ensure that the results are reproducable.

In [3]:
Auto = load_data('Auto')
n = len(Auto)
Auto_train, Auto_val = train_test_split(
    Auto,
    test_size = int(n / 2),
    random_state = 0
)

With the testing and training sets established, a linear regression model can be fit on the training data.

In [4]:
hp = MS(['horsepower'])
X_train = hp.fit_transform(Auto_train)
y_train = Auto_train['mpg']
model = sm.OLS(y_train, X_train)
results = model.fit()

Now, the results of the model can be tested on the validation (testing) data set. Additionally, the MSE of the model can be calculated.

In [5]:
X_val = hp.fit_transform(Auto_val)
y_val = Auto_val['mpg']
val_pred = results.predict(X_val)
np.mean((y_val - val_pred) ** 2)

np.float64(23.61661706966988)

As can be seen, the validation MSE is roughly 23.6 using linear regression on this validation set. The validation MSE can also be tested using a higher-order polynomial regression. To do so, a function **eval_MSE()** must be defined to take a model string and training and test sets and returns the MSE on the test set. 

In [6]:
def eval_MSE(terms, response, train, test):
    mn = MS(terms)
    X_train = mn.fit_transform(train)
    y_train = train[response]
    X_test = mn.transform(test)
    y_test = test[response]
    
    model = sm.OLS(y_train, X_train).fit()
    results = model.predict(X_test)
    return np.mean((y_test - results) ** 2)

This function will be used to estimate the validation MSE using linear, quadratic, and cubic fits. 

In [7]:
MSE = np.zeros(3)
for i, v in enumerate(range(1, 4)):
    MSE[i] = eval_MSE(
        [poly('horsepower', v)],
        'mpg',
        Auto_train,
        Auto_val
    )
MSE

array([23.61661707, 18.76303135, 18.79694163])

The resulting MSEs are roughly 23.62, 18.76, and 18.77 for linear, quadratic, and cubic approaches, respectively. Based on the results, it appears that the quadratic approach is most suitable (though cubic is very close). If a different training and testing split were used, the results could be expected to be a bit different.

In [8]:
Auto_train, Auto_val = train_test_split(
    Auto, 
    test_size=196, 
    random_state = 3
)
MSE = np.zeros(3)
for i, v in enumerate(range(1, 4)):
    MSE[i] = eval_MSE(
        [poly('horsepower', v)],
        'mpg',
        Auto_train,
        Auto_val
    )
MSE

array([20.75540796, 16.94510676, 16.97437833])

The above results are consistent with the first test of linear vs. quadratic vs. cubic models. The quadratic model is a significant improvement ove the linear model with little difference between quadratic and cubic models.

### Cross-Validation
In theory, cross-validation can be computed for any generalized-linear model. However, the simplest way to perform cross-validation in Python is to use the **sklearn** library, which has a different API than the **statsmodels** library that has been used to this point in fitting GLMs.

This is a common problem where a tool exists to solve one problem, but another tool is needed to solve a problem on top of that problem. Such instances require a wrapper function that allows the results of the first tool to be passed to the next. Luckily, the **ISLP** library includes the **sklearn_sm()** wrapper class that enables the easy use of cross-validation tools from **sklearn** with models fit by **statsmodels**.

The **sklearn_sm()** class takes the **statsmodels** model as its first argument and can take two additional arguments:
1. **model_str**: Which can be used to specify a formula.
2. **model_args**: Which should be a dictionary of additional arguments used when fitting the model.

For example, to specify a logistic regression model the **family** argument must be specified. To do so, pass in *model_args={'family':sm.families.Binomial()}*. Below, the wrapper is demonstrated/

In [9]:
hp_model = sklearn_sm(sm.OLS, MS(['horsepower']))
X, Y = Auto.drop(columns=['mpg']), Auto['mpg']
cv_results = cross_validate(
    hp_model,
    X, 
    Y,
    cv=Auto.shape[0]
)
cv_error = np.mean(cv_results['test_score'])
cv_error

np.float64(24.23151351792922)

The arguments to **cross_validate** are as follows:
1. The model to validate (must have **fit()**, **predict()**, and **score()** methods).
2. An array of features.
3. An array for the response.
4. **cv** an optional argument which takes an integer specifying the number folds to perform.

In the case above, the value of **cv** was set to the total number of observation. This results in *leave-one-out cross-validation* (LOOCV). 

The **cross_validate** function produces a dictionary with several components. The **test_score** component corresponds to the test MSE. This process can be repeated for increasingly complex polynomial fits, as was done with MSE in the previous section.

In [10]:
cv_errors = np.zeros(5)
H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)
for i, d in enumerate(range(1, 6)):
    X = np.power.outer(H, np.arange(d + 1))
    M_CV = cross_validate(
        M,
        X,
        Y,
        cv=Auto.shape[0]
    )
    cv_errors[i] = np.mean(M_CV['test_score'])
cv_errors

array([24.23151352, 19.24821312, 19.33498406, 19.42443031, 19.03321178])

As with the polynomial testing without cross-validation, the improvement over linear regression is significant when moving to a quadratic model, but improvements with cubic and higher-degree approaches are minimal when compared to quadratic.

In the code cell above, the **np.power().outer()** method is used. This method is applied to an operation that has two arguments (ex. **add()**, **min()**, or **power()**). It takes two arrays as arguments and returns a larger array where the operation is applied to each pair of elements of the two arrays. For example, using **outer()** on an **add()** on a 3-element array A and a 2-element array B will result in a 3x2 array (as shown below).

In [11]:
A = np.array([3, 5, 9])
B = np.array([2, 4])
C = np.add.outer(A, B)
C, C.shape

(array([[ 5,  7],
        [ 7,  9],
        [11, 13]]),
 (3, 2))

Of course, there is no need to use LOOCV in every case. A $K$ can be chosen such that $K < n$, and, in fact, this will be much quicker (especially if $K << n$). Below, an example with $K = 10$ is produced using the **KFold()** function to produce random groups. A random seed is used to ensure replicability of the results.

In [12]:
cv_errors = np.zeros(5)
cv = KFold(
    n_splits = 10,
    shuffle = True,
    random_state = 0
)
for i, d in enumerate(range(1, 6)):
    X = np.power.outer(H, np.arange(d + 1))
    M_CV = cross_validate(
        M,
        X,
        Y,
        cv=cv
    )
    cv_errors[i] = np.mean(M_CV['test_score'])
cv_errors

array([24.20766449, 19.18533142, 19.27626666, 19.47848402, 19.13720959])

While this is faster in practice, it is useful to note that in an optimized case the availability of a formula for LOOCV means that the previous approach should, in theory, actually be faster. However, the generic **cross_validate()** function does not make use of this formula.

Again, there is little evidence to suggest that the higher-degree polynomial terms above quadratci provide any meaningful improvement to the model.

The **cross_validate()** function is quite flexible and can take various splitting mechanisms as arguments (not just **KFold()**). This can be seen below with the **ShuffleSplit()** function.

In [13]:
validation = ShuffleSplit(
    n_splits = 1,
    test_size = 196,
    random_state = 0
)
results = cross_validate(
    hp_model,
    Auto.drop(['mpg'], axis=1),
    Auto['mpg'],
    cv=validation
)
error = np.mean(results['test_score'])
error

np.float64(23.61661706966988)

The variablity in the test error can be examined using the **np.std()** on the **test_score** component of the cross validation results.

In [14]:
validation = ShuffleSplit(
    n_splits = 10,
    test_size = 196,
    random_state=0
)
results = cross_validate(
    hp_model,
    Auto.drop(columns=['mpg']),
    Auto['mpg'],
    cv = validation
)
print("Test Error:", np.mean(results['test_score']))
print("Test Std. Dev.:", results['test_score'].std())

Test Error: 23.802232661034168
Test Std. Dev.: 1.4218450941091842


The book notes that this standard deviation is not a true representation of the sampling variability of the mean test score or the individual scores, since that training samples have overlap. This overlap introduces some correlation in the training samples. However, it gives an idea of the Monte Carlo variation incurred by picking different random folds.

### The Bootstrap
In this section, the bootstrap is demonstrated on the example found in section 5.2 of the text. Additionally, an example involving estimating the accuracy of a linear regression model is used.

###### Estimating the Accuracy of a Statistic of Interest
One of the great benefits of the bootstrap is that it can be applied to just about any case, and no complicated mathematical calculations are required. Several methods exist for performing the boostrap in Python, but due to its simple nature, it is implemented manually in this lab.

To begin, a simple example on the **Portfolio** data set is used. The goal is to estimate the sampling variance of the parameter $\alpha$ given in the formula 5.7 of the text. In pusuit of this goal, a **alpha_func()** function is created to take a dataframe with columns **X** and **Y** as well as an **idx** vector indication which observations to use in estimating $\alpha$. The function returns the estimated value of $\alpha$ based on the selected observations.

In [15]:
Portfolio = load_data('Portfolio')
def alpha_func(D, idx):
    cov_ = np.cov(D[['X', 'Y']].loc[idx], rowvar=False)
    num = (cov_[1,1] - cov_[0,1])
    den = (cov_[0,0] + cov_[1,1] - 2 * cov_[0,1])
    return num / den

The above formula returns an estimate for $\alpha$ based on the minimum variance formula to the observations indexed by the argument **idx**. For example, the code below estimates $\alpha$ based on all 100 observations.

In [16]:
alpha_func(Portfolio, range(100))

np.float64(0.57583207459283)

The next step for producing a boostrapped standard error is to select randomized data sets picked form the original dataset with replacement. That is duplicate observations are permitted. The **boot_SE()** function is defined below utilizing such random selection.

In [19]:
def boot_SE(func, D, n=None, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(
            D.index,
            n,
            replace=True
        )
        value = func(D, idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_ / B - (first_ / B)**2)

This function can now be used to estimate the accuracy of $\alpha$ obtained above.

In [20]:
alpha_SE = boot_SE(alpha_func, Portfolio)
alpha_SE

np.float64(0.09118176521277699)

This output indicates that the bootstrap estimate for $SE(\alpha)$ is 0.0912.